# SNPs Extaction and Mapping

In [2]:
import os
import glob
import numpy as np
import pandas as pd
from cyvcf2 import VCF

In [ ]:
path = "/Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/tuberculosis"

WHO_FILE = f"{path}/who_mutations.csv"  # WHO mutation list
VCF_PATTERN = f"{path}/assemblies.nosync/*.raw.vcf.gz"  # VCF file pattern
OUTPUT = f"{path}/strains_snps.csv"  # Output SNP side information

In [8]:
print("Loading WHO mutation list...")
who = pd.read_csv(WHO_FILE, dtype={"pos": int})
required = {"mut_name", "pos", "ref", "alt"}
if not required.issubset(who.columns):
    raise ValueError(f"who_mutations.csv must contain columns: {required}")

# 去重（安全）
who = who.drop_duplicates(subset=["mut_name"])
mut_names = who["mut_name"].tolist()
print(f"Loaded {len(mut_names)} WHO mutations.")

Loading WHO mutation list...
Loaded 118431 WHO mutations.


In [9]:
# 查找所有 VCF
vcf_files = sorted(glob.glob(VCF_PATTERN))
if len(vcf_files) == 0:
    raise RuntimeError(f"No VCF files found matching: {VCF_PATTERN}")

print(f"Found {len(vcf_files)} VCF files.")

Found 7932 VCF files.


In [ ]:
rows = []
index = []
# -------------------- 主循环：对每个 assembly 处理一次 --------------------
for vcf_path in vcf_files:
    acc = os.path.basename(vcf_path).replace(".raw.vcf.gz", "")
    print(f"Processing {acc} ...")

    # 单个 VCF 的突变结果（初始为 NaN）
    row = {m: np.nan for m in mut_names}

    v = VCF(vcf_path)

    # 建立位置索引：pos → list(records)
    pos_dict = {}
    for rec in v:
        pos_dict.setdefault(int(rec.POS), []).append(rec)

    # 针对每个 WHO 突变
    for _, w in who.iterrows():
        pos = int(w["pos"])
        ref_w = str(w["ref"])
        alt_w = str(w["alt"])
        mut = w["mut_name"]

        recs = pos_dict.get(pos, [])
        if len(recs) == 0:
            # 没有记录 → 无法调用
            row[mut] = np.nan
            continue

        seen = False

        for rec in recs:
            # REF / ALT 匹配
            rec_ref = rec.REF
            rec_alts = [str(a) for a in rec.ALT]

            if rec_ref == ref_w and alt_w in rec_alts:
                seen = True

        row[mut] = 1 if seen else 0

    rows.append(row)
    index.append(acc)

In [11]:
# -------------------- 保存为矩阵 --------------------
mat = pd.DataFrame(rows, index=index)
mat.index.name = "assembly_acc"
mat = mat.reindex(columns=mut_names)

In [77]:
mat.iloc[:10, :3]

,NC_000962.3_8_A>C,NC_000962.3_8_AT>CA,NC_000962.3_8_AT>CC
assembly_acc,,,
GCA_000008585.1,NaN,NaN,NaN
GCA_000016145.1,NaN,NaN,NaN
GCA_000016925.1,NaN,NaN,NaN
GCA_000023625.1,NaN,NaN,NaN
GCA_000154605.2,NaN,NaN,NaN
GCA_000155185.1,NaN,NaN,NaN
GCA_000159735.1,NaN,NaN,NaN
GCA_000159755.1,NaN,NaN,NaN
GCA_000162995.1,NaN,NaN,NaN


In [ ]:
print(f"Writing: {OUTPUT} shape={mat.shape}")
mat.to_csv(OUTPUT)
print("Done.")

---

## Screen of the processed SNPs

### 1. Remove columns with all NaN or 0's. 

In [12]:
not_all_nan = mat.notna().any(axis=0)

In [19]:
sum(not_all_nan)

58655

In [13]:
not_all_zero = (mat.fillna(0) != 0).any(axis=0)

In [18]:
sum(not_all_zero)

9684

In [14]:
keep_cols = not_all_nan & not_all_zero

In [112]:
mat_screened = mat.loc[:, keep_cols]
print("Original SNPs:", mat.shape[1])
print("After screened SNPs:", mat_screened.shape[1])
print("Total removed SNPs:", mat.shape[1] - mat_screened.shape[1])

Original SNPs: 118431
After screened SNPs: 9684
Total removed SNPs: 108747


In [41]:
mat_screened.columns

Index(['NC_000962.3_11_A>C', 'NC_000962.3_29_C>G', 'NC_000962.3_31_A>C',
       'NC_000962.3_32_C>A', 'NC_000962.3_33_A>G', 'NC_000962.3_44_C>T',
       'NC_000962.3_64_G>C', 'NC_000962.3_71_C>T', 'NC_000962.3_78_T>G',
       'NC_000962.3_82_G>C',
       ...
       'NC_000962.3_4411419_G>A', 'NC_000962.3_4411420_G>A',
       'NC_000962.3_4411438_G>A', 'NC_000962.3_4411440_T>C',
       'NC_000962.3_4411475_G>A', 'NC_000962.3_4411485_G>C',
       'NC_000962.3_4411500_C>T', 'NC_000962.3_4411501_C>T',
       'NC_000962.3_4411518_C>T', 'NC_000962.3_4411522_G>A'],
      dtype='object', length=9684)

In [ ]:
mat_screened = mat_screened.fillna(0).astype(int)
mat_screened.head()

,NC_000962.3_11_A>C,NC_000962.3_29_C>G,NC_000962.3_31_A>C,NC_000962.3_32_C>A,NC_000962.3_33_A>G,NC_000962.3_44_C>T,NC_000962.3_64_G>C,NC_000962.3_71_C>T,NC_000962.3_78_T>G,NC_000962.3_82_G>C,...,NC_000962.3_4411419_G>A,NC_000962.3_4411420_G>A,NC_000962.3_4411438_G>A,NC_000962.3_4411440_T>C,NC_000962.3_4411475_G>A,NC_000962.3_4411485_G>C,NC_000962.3_4411500_C>T,NC_000962.3_4411501_C>T,NC_000962.3_4411518_C>T,NC_000962.3_4411522_G>A
assembly_acc,,,,,,,,,,,,,,,,,,,,,
GCA_000008585.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000016145.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000016925.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000023625.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000154605.2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. Use WHO information to keep only drug resistance-related SNPs

In [ ]:
WHO_XLSX = (
    f"{path}/mutation-catalogue-2023/Final Result Files/WHO-UCN-TB-2023.7-eng.xlsx"
)
# load WHO sheets
genomic = pd.read_excel(WHO_XLSX, sheet_name="Genomic_coordinates", dtype=str)
catalog = pd.read_excel(
    WHO_XLSX, sheet_name="Catalogue_master_file", dtype=str, header=2
)

In [95]:
genomic.head()

,variant,chromosome,position,reference_nucleotide,alternative_nucleotide,variant_short,snp_id
0,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CA,p.Asp3Ala,NC_000962.3_8_AT>CA
1,dnaA_p.Asp3Ala,NC_000962.3,8,A,C,p.Asp3Ala,NC_000962.3_8_A>C
2,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CC,p.Asp3Ala,NC_000962.3_8_AT>CC
3,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CG,p.Asp3Ala,NC_000962.3_8_AT>CG
4,dnaA_p.Asp4His,NC_000962.3,10,G,C,p.Asp4His,NC_000962.3_10_G>C


In [96]:
catalog.head()

,drug,gene,mutation,variant,tier,effect,genomic position,algorithm_pass,Present_SOLO_SR,Present_SOLO_R,...,Additional grading criteria applied,FINAL CONFIDENCE GRADING,Comment,CHANGES vs ver1,"Relaxed thresholds simulation (BDQ_Rv0678, CFZ_Rv0678, INH_katG, DLM_ddn/fbiA/fbiB/fbiC/fgd1/Rv2983)",Silent mutation,Listed in abridged tables,Additional grading,Footnote,CHANGES vs ver1.1
0,Amikacin,bacA,c.102G>A,bacA_c.102G>A,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,Silent mutation,4) Not assoc w R - Interim,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0
1,Amikacin,bacA,c.1044G>A,bacA_c.1044G>A,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,NaN,5) Not assoc w R,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0
2,Amikacin,bacA,c.105C>G,bacA_c.105C>G,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,Silent mutation,4) Not assoc w R - Interim,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0
3,Amikacin,bacA,c.1065T>G,bacA_c.1065T>G,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,Silent mutation,4) Not assoc w R - Interim,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0
4,Amikacin,bacA,c.1080G>A,bacA_c.1080G>A,2,synonymous_variant,"(see ""Genomic_coordinates"" sheet)",NaN,NaN,NaN,...,Silent mutation,4) Not assoc w R - Interim,NaN,Now listed,NaN,Silent mutation,no,NaN,NaN,0


In [ ]:
def strip_prefix(v):
    if pd.isna(v):
        return v
    return re.sub(r"^[^_]+_", "", v)

In [84]:
genomic["variant_short"] = genomic["variant"].apply(strip_prefix)
genomic.head()

,variant,chromosome,position,reference_nucleotide,alternative_nucleotide,variant_short
0,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CA,p.Asp3Ala
1,dnaA_p.Asp3Ala,NC_000962.3,8,A,C,p.Asp3Ala
2,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CC,p.Asp3Ala
3,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CG,p.Asp3Ala
4,dnaA_p.Asp4His,NC_000962.3,10,G,C,p.Asp4His


In [94]:
genomic["snp_id"] = (
    genomic["chromosome"]
    + "_"
    + genomic["position"]
    + "_"
    + genomic["reference_nucleotide"]
    + ">"
    + genomic["alternative_nucleotide"]
)
genomic.head()

,variant,chromosome,position,reference_nucleotide,alternative_nucleotide,variant_short,snp_id
0,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CA,p.Asp3Ala,NC_000962.3_8_AT>CA
1,dnaA_p.Asp3Ala,NC_000962.3,8,A,C,p.Asp3Ala,NC_000962.3_8_A>C
2,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CC,p.Asp3Ala,NC_000962.3_8_AT>CC
3,dnaA_p.Asp3Ala,NC_000962.3,8,AT,CG,p.Asp3Ala,NC_000962.3_8_AT>CG
4,dnaA_p.Asp4His,NC_000962.3,10,G,C,p.Asp4His,NC_000962.3_10_G>C


In [100]:
genomic_drug_screened = genomic[genomic["variant"].isin(catalog["variant"])].copy()
print("The drug resistance-related SNPs in WHO:", genomic_drug_screened.shape[0])

The drug resistance-related SNPs in WHO: 124726


In [103]:
mat_drug_screened = mat_screened[
    mat_screened.columns.intersection(genomic_drug_screened["snp_id"])
].copy()
print("The drug resistance SNPs in our dataset:", mat_drug_screened.shape[1])

The drug resistance SNPs in our dataset: 9684


In [ ]:
OUTPUT = f"{path}/strains_snps_screened.csv"
mat_drug_screened.to_csv(OUTPUT)

In [105]:
mat_drug_screened.head()

,NC_000962.3_11_A>C,NC_000962.3_29_C>G,NC_000962.3_31_A>C,NC_000962.3_32_C>A,NC_000962.3_33_A>G,NC_000962.3_44_C>T,NC_000962.3_64_G>C,NC_000962.3_71_C>T,NC_000962.3_78_T>G,NC_000962.3_82_G>C,...,NC_000962.3_4411419_G>A,NC_000962.3_4411420_G>A,NC_000962.3_4411438_G>A,NC_000962.3_4411440_T>C,NC_000962.3_4411475_G>A,NC_000962.3_4411485_G>C,NC_000962.3_4411500_C>T,NC_000962.3_4411501_C>T,NC_000962.3_4411518_C>T,NC_000962.3_4411522_G>A
assembly_acc,,,,,,,,,,,,,,,,,,,,,
GCA_000008585.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000016145.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000016925.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000023625.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000154605.2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


---

### Future annotation work

- The WHO file provides direct SNP information to drugs, and genes, effects and etc..

- Use the following parts as inspiration for annotation work after selection of SNPs from the algorithm. 

In [109]:
map_key2variant = genomic.set_index("snp_id")["variant_short"]

In [110]:
map_key2variant.head(10)

snp_id
NC_000962.3_8_AT>CA       p.Asp3Ala
NC_000962.3_8_A>C         p.Asp3Ala
NC_000962.3_8_AT>CC       p.Asp3Ala
NC_000962.3_8_AT>CG       p.Asp3Ala
NC_000962.3_10_G>C        p.Asp4His
NC_000962.3_10_GAC>CAT    p.Asp4His
NC_000962.3_11_AC>CG      p.Asp4Ala
NC_000962.3_11_A>C        p.Asp4Ala
NC_000962.3_11_AC>CA      p.Asp4Ala
NC_000962.3_11_AC>CT      p.Asp4Ala
Name: variant_short, dtype: object

In [ ]:
mat_annotated = mat_drug_screened.rename(columns=map_key2variant)

In [72]:
print(mat_annotated.columns)

Index(['p.Asp4Ala', 'p.Thr10Ser', 'p.Thr11Pro', 'p.Thr11Lys', 'c.33A>G',
       'p.Ala15Val', 'p.Gly22Arg', 'p.Pro24Leu', 'c.78T>G', 'p.Asp28His',
       ...
       'c.-114G>A', 'c.-113G>A', 'c.-95G>A', 'c.-93T>C', 'c.-58G>A',
       'c.-48G>C', 'c.-33C>T', 'c.-32C>T', 'c.-15C>T', 'c.-11G>A'],
      dtype='object', length=9684)
